In [ ]:
from dotenv import load_dotenv

load_dotenv()

from typing import Annotated
from typing_extensions import TypedDict

from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage
from langchain_core.messages import SystemMessage, RemoveMessage
from langchain_core.messages.utils import count_tokens_approximately

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver

In [ ]:
# ============================================================
# 1. CREATE LLM
# ============================================================

model = init_chat_model(
    "openai:gpt-5.4-nano",
    temperature=0
)


# ============================================================
# 2. DEFINE LIMITS
# ============================================================

TOKEN_LIMIT = 2000
MESSAGE_LIMIT = 6


# ============================================================
# 3. DEFINE STATE
# ============================================================

class State(TypedDict):

    messages: Annotated[
        list,
        add_messages
    ]

    summary: str

In [ ]:
# ============================================================
# 4. DEFINE SUMMARIZATION NODE
# ============================================================

def summarize_node(state: State):

    messages = state["messages"]
    summary = state.get("summary", "")

    token_count = count_tokens_approximately(messages)

    print(f"\n[summarize_node] messages passed to LLM: {len(messages)}")
    print(f"[summarize_node] tokens passed to LLM: {token_count}")

    if token_count <= TOKEN_LIMIT and len(messages) <= MESSAGE_LIMIT:
        return {}

    print("[summarize_node] limit exceeded -> summarizing older messages")

    if summary:
        summary_prompt = (
            f"This is the summary of the conversation so far: {summary}\n\n"
            "Extend the summary by taking into account the new messages above:"
        )
    else:
        summary_prompt = "Create a summary of the conversation above:"

    messages_to_summarize = messages[:-1]

    summary_response = model.invoke(
        messages_to_summarize + [HumanMessage(content=summary_prompt)]
    )

    delete_messages = [
        RemoveMessage(id=m.id) for m in messages_to_summarize
    ]

    return {
        "summary": summary_response.content,
        "messages": delete_messages
    }

In [ ]:
# ============================================================
# 5. DEFINE CHATBOT NODE
# ============================================================

def chatbot(state: State):

    summary = state.get("summary", "")
    messages = state["messages"]

    if summary:
        messages_for_llm = [
            SystemMessage(
                content=f"Summary of earlier conversation: {summary}"
            )
        ] + messages
    else:
        messages_for_llm = messages

    token_count = count_tokens_approximately(messages_for_llm)
    print(f"[chatbot] tokens actually sent to LLM: {token_count}")

    response = model.invoke(messages_for_llm)

    return {
        "messages": [response]
    }

In [ ]:
# ============================================================
# 6. BUILD GRAPH
# ============================================================

builder = StateGraph(State)

builder.add_node(
    "summarize",
    summarize_node
)

builder.add_node(
    "chatbot",
    chatbot
)

builder.add_edge(
    START,
    "summarize"
)

builder.add_edge(
    "summarize",
    "chatbot"
)

builder.add_edge(
    "chatbot",
    END
)

In [ ]:
# ============================================================
# 7. CREATE CHECKPOINTER
# ============================================================

checkpointer = InMemorySaver()


# ============================================================
# 8. COMPILE GRAPH WITH CHECKPOINTER
# ============================================================

graph = builder.compile(
    checkpointer=checkpointer
)

In [ ]:
from IPython.display import Image, display

display(
    Image(
        graph.get_graph().draw_mermaid_png()
    )
)

In [ ]:
# ============================================================
# 9. CREATE THREAD
# ============================================================

config = {
    "configurable": {
        "thread_id": "thread-001"
    }
}

In [ ]:
# ============================================================
# 10. DEMO A - MESSAGE 1
# ============================================================

result = graph.invoke(
    {
        "messages": [
            HumanMessage(
                content="Hi, my name is Rahul."
            )
        ]
    },
    config
)

print("\nAI:", result["messages"][-1].content)

In [ ]:
# ============================================================
# 11. DEMO A - MESSAGE 2
# ============================================================

result = graph.invoke(
    {
        "messages": [
            HumanMessage(
                content="I live in Bangalore."
            )
        ]
    },
    config
)

print("\nAI:", result["messages"][-1].content)

In [ ]:
# ============================================================
# 12. DEMO A - MESSAGE 3
# ============================================================

result = graph.invoke(
    {
        "messages": [
            HumanMessage(
                content="I work as a data engineer."
            )
        ]
    },
    config
)

print("\nAI:", result["messages"][-1].content)

In [ ]:


# ============================================================
# 13. DEMO A - MESSAGE 4 (MESSAGE-COUNT LIMIT SHOULD BE CROSSED HERE)
# ============================================================

result = graph.invoke(
    {
        "messages": [
            HumanMessage(
                content="My favorite hobby is playing cricket."
            )
        ]
    },
    config
)

print("\nAI:", result["messages"][-1].content)

In [ ]:
# ============================================================
# 14. DEMO A - VIEW STATE AFTER MESSAGE 4
# ============================================================

state_a = graph.get_state(config)

print("\nsummary:", state_a.values.get("summary"))
print("\nmessages kept:", len(state_a.values["messages"]))

for m in state_a.values["messages"]:
    m.pretty_print()

In [ ]:
# ============================================================
# 15. DEMO A - MESSAGE 5 (WHAT IS MY NAME)
# ============================================================

result = graph.invoke(
    {
        "messages": [
            HumanMessage(
                content="What is my name and where do I live?"
            )
        ]
    },
    config
)

print("\nAI:", result["messages"][-1].content)

In [ ]:
# ============================================================
# 16. DEMO A - VIEW STATE AFTER MESSAGE 5
# ============================================================

state_a = graph.get_state(config)

print("\nsummary:", state_a.values.get("summary"))
print("\nmessages kept:", len(state_a.values["messages"]))

In [ ]:
# ============================================================
# 17. DEMO B - CREATE SECOND THREAD
# ============================================================

config2 = {
    "configurable": {
        "thread_id": "thread-002"
    }
}

In [ ]:
# ============================================================
# 18. DEMO B - LONG MESSAGE (TOKEN-COUNT LIMIT TRIGGER)
# ============================================================

long_text = (
    "This is a filler sentence used only to inflate the token count for this demo. "
    * 400
)

result = graph.invoke(
    {
        "messages": [
            HumanMessage(
                content=long_text
            )
        ]
    },
    config2
)

print("\nAI:", result["messages"][-1].content)

In [ ]:
# ============================================================
# 19. DEMO B - VIEW STATE AFTER TOKEN-TRIGGERED SUMMARIZATION
# ============================================================

state_b = graph.get_state(config2)

print("\nsummary:", state_b.values.get("summary"))
print("\nmessages kept:", len(state_b.values["messages"]))

In [ ]:
# ============================================================
# 20. PRETTY PRINT FINAL MESSAGES (THREAD 1)
# ============================================================

final_state = graph.get_state(config)

for m in final_state.values["messages"]:
    m.pretty_print()